# 🧠 CareerCompass AI Engine: Global Skill NER training (Autonomous)

This notebook trains a custom NER model to extract skills and roles. 

**Strategy:** We utilize a **Synthetic Data Augmentation** engine to bootstrap our training. This avoids dependencies on private/gated external datasets and ensures 100% reliability for the graduation project.

## 📝 Step 1: Environment Setup & Dependencies Installation

In [ ]:
!pip install transformers datasets seqeval evaluate torch accelerate -U

## 📝 Step 2: Data Loading & Label Configuration

In [ ]:
import json
from datasets import load_dataset
import sys
import os
# sys.path.append(os.path.abspath('..'))
dataset_path = "/content/train_real_tech_cleaned.json"

# 1. Define Labels (Including Soft Skills)
label_list = ["O", "B-SKILL", "I-SKILL", "B-ROLE", "I-ROLE", "B-EDU", "I-EDU", "B-CERT", "I-CERT", "B-SOFT", "I-SOFT"]

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print(f"Total labels: {len(label_list)}")

# 2. Load and Split Dataset
dataset = load_dataset('json', data_files='train_real_tech_cleaned.json')
dataset = dataset['train'].train_test_split(test_size=0.1, seed=42)

print(dataset)

## 📝 Step 3: Tokenization & Label Alignment

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=512,
        return_offsets_mapping=True
    )

    labels = []
    for i, entity_list in enumerate(examples["entities"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        offsets = tokenized_inputs["offset_mapping"][i]
        label_ids = [0] * len(word_ids)

        seen_chars = set()
        for ent in entity_list:
            ent_text = ent["text"]
            ent_label = ent["label"]
            start_char = examples["text"][i].find(ent_text)
            if start_char == -1: continue
            end_char = start_char + len(ent_text)
            
            b_label = label2id.get(f"B-{ent_label}", 0)
            i_label = label2id.get(f"I-{ent_label}", 0)
            
            first = True
            for idx, (tok_start, tok_end) in enumerate(offsets):
                if tok_start == tok_end: continue # special token
                if tok_start >= start_char and tok_end <= end_char:
                    if first:
                        label_ids[idx] = b_label
                        first = False
                    else:
                        label_ids[idx] = i_label
                    seen_chars.add(idx)

        for idx, wid in enumerate(word_ids):
            if wid is None:
                label_ids[idx] = -100

        labels.append(label_ids)
    tokenized_inputs.pop("offset_mapping")
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
print("✅ Tokenization and Offset Mapping Complete!")


## Step 4: Model Initialization & Metrics Definition

In [ ]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
import evaluate
import numpy as np

# Initialize Model with ignore_mismatched_sizes
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## 📝 Step 5: Model Training & Evaluation

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=1000,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="steps",
    save_steps=1000,
    load_best_model_at_end=True,
    fp16=True,
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,  
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)


## 📝 Step 6: Model Export & Download

In [ ]:
from google.colab import files
import os

output_dir = "career_compass_ner_final"

# 1. Save Model & Tokenizer
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ Model saved successfully to {output_dir}/")

# 2. Compress Directory
zip_name = f"{output_dir}.zip"
os.system(f"zip -r {zip_name} {output_dir}/")
print("✅ Folder zipped successfully!")

# 3. Download to Local Machine
files.download(zip_name)